In [6]:
import numpy as np
import pandas as pd

df = pd.read_csv('raw.csv')

df = df[['userId', 'productId', 'score']].dropna()

print(df.shape)   
df.head()

(252056, 3)


,userId,productId,score
0,A1FWT811DSZLC8,B00064C0IU,3.0
1,A1THE6V6O8ROD4,B00064C0IU,5.0
2,A176IQ7MVD3N6T,B00064C0IU,1.0
3,A34BDX4JVMG23Y,B00064C0IU,5.0
4,A3UWJXJI7S3T05,B000K5JBZU,5.0


In [2]:
# Giữ user có ít nhất 5 reviews
# Giữ product có ít nhất 10 reviews

user_counts    = df['userId'].value_counts()
product_counts = df['productId'].value_counts()

df_filtered = df[
    df['userId'].isin(user_counts[user_counts >= 5].index) &
    df['productId'].isin(product_counts[product_counts >= 10].index)
]

print(f"Còn lại: {df_filtered['userId'].nunique()} users, "
      f"{df_filtered['productId'].nunique()} products")

Còn lại: 3923 users, 3936 products


In [3]:
from sklearn.model_selection import train_test_split

# Sử dụng df_filtered để tránh MemoryError khi tạo ma trận rating dense
train, test = train_test_split(df_filtered, test_size=0.2, random_state=42)

print(f"Train: {len(train)} | Test: {len(test)}")

Train: 28209 | Test: 7053


In [4]:
# Tạo index mapping từ df_filtered
users    = list(df_filtered['userId'].unique())
products = list(df_filtered['productId'].unique())

user2idx    = {u: i for i, u in enumerate(users)}
product2idx = {p: j for j, p in enumerate(products)}

# Xây ma trận R từ train set
R = np.zeros((len(users), len(products)))

for _, row in train.iterrows():
    i = user2idx[row['userId']]
    j = product2idx[row['productId']]
    R[i][j] = row['score']

print(f"Ma trận shape: {R.shape}")
print(f"Độ thưa: {(R == 0).sum() / R.size * 100:.1f}%")

Ma trận shape: (3923, 3936)
Độ thưa: 99.8%


In [5]:
# Tính mean rating của mỗi user
user_mean = {}
for _, row in train.iterrows():
    u = user2idx[row['userId']]
    if u not in user_mean:
        user_mean[u] = []
    user_mean[u].append(row['score'])

user_mean = {u: np.mean(scores) for u, scores in user_mean.items()}

# Xây ma trận R đã center
R_centered = np.zeros((len(users), len(products)))

for _, row in train.iterrows():
    i = user2idx[row['userId']]
    j = product2idx[row['productId']]
    R_centered[i][j] = row['score'] - user_mean[i]  # trừ mean

# SVD trên R_centered
U, S, Vt = np.linalg.svd(R_centered, full_matrices=False)

# Hàm dự đoán xếp hạng rút gọn
def predict(U, S, Vt, k):
    return np.dot(U[:, :k] * S[:k], Vt[:k, :])

# Dự đoán → cộng mean lại
def predict_centered(U, S, Vt, k, user_mean):
    R_pred = predict(U, S, Vt, k)
    for i in range(R_pred.shape[0]):
        R_pred[i] += user_mean.get(i, 3.0)  # cộng mean user, mặc định 3.0
    # Clip về [1, 5]
    return np.clip(R_pred, 1.0, 5.0)

# Định nghĩa hàm RMSE để đánh giá độ lỗi
def rmse(R_pred, test_df, user2idx, product2idx):
    sq_errors = []
    for _, row in test_df.iterrows():
        u = row['userId']
        p = row['productId']
        actual = row['score']
        if u in user2idx and p in product2idx:
            i = user2idx[u]
            j = product2idx[p]
            pred = R_pred[i][j]
            sq_errors.append((actual - pred) ** 2)
    return np.sqrt(np.mean(sq_errors)) if sq_errors else 0.0

# Đánh giá lại với các K khác nhau
for k in [5, 10, 20, 50]:
    R_pred = predict_centered(U, S, Vt, k, user_mean)
    score  = rmse(R_pred, test, user2idx, product2idx)
    print(f"K={k:3d} → RMSE: {score:.4f}")

K=  5 → RMSE: 0.7992


K= 10 → RMSE: 0.7958


K= 20 → RMSE: 0.7933


K= 50 → RMSE: 0.7947


In [6]:
def recommend(user_id, R, R_pred, user2idx, products, top_n=5):
    if user_id not in user2idx:
        print("User mới, chưa có data!")
        return []

    u_idx = user2idx[user_id]

    # Lọc sp chưa mua
    recommendations = []
    for j, product in enumerate(products):
        if R[u_idx][j] == 0:  # chưa rating
            recommendations.append((product, R_pred[u_idx][j]))

    # Sort theo score dự đoán
    recommendations.sort(key=lambda x: x[1], reverse=True)
    return recommendations[:top_n]

# Thử với 1 user mẫu từ tập dữ liệu
sample_user = df_filtered['userId'].iloc[0]
R_pred = predict_centered(U, S, Vt, k=10, user_mean=user_mean)
recs   = recommend(sample_user, R, R_pred, user2idx, products)

print(f"Gợi ý cho User: {sample_user}")
for product_id, score in recs:
    print(f"{product_id} → predicted score: {score:.2f}")

Gợi ý cho User: A21HOY50DPUYQN
B000NP6EWK → predicted score: 5.00
B000BR15GK → predicted score: 5.00
B0001EKYJ2 → predicted score: 5.00
B0001EKYLA → predicted score: 5.00
B0001EKYL0 → predicted score: 5.00
